# Notebook 02 — Activation Extraction: Layer-Wise Hidden States

**TIN-7: Cross-Lingual Embedding Alignment Analysis**

This notebook loads Tiny Aya Global, extracts mean-pooled sentence
embeddings from all 4 transformer layers for all 13 languages, saves
them to disk for reuse, and visualizes the embedding spaces using
dimensionality reduction (PCA, t-SNE).

## What this notebook does

1. Loads `CohereLabs/tiny-aya-global` (fp16 by default, 4-bit optional).
2. Processes all 1,012 FLORES-200 sentences per language through the
   model, extracting hidden states at every transformer layer.
3. Mean-pools over non-padding tokens to produce sentence-level
   embeddings of shape `(1012, 3072)` per language per layer.
4. Saves activations to disk as `.pt` files with standardized keys.
5. Visualizes embeddings with t-SNE/PCA colored by language.

## Prerequisites

- GPU with >= 8GB VRAM (for fp16) or >= 2GB (for 4-bit).
- Run Notebook 01 first to verify data loading works.

## Estimated runtime

- ~10-15 minutes on RTX 2070 (8GB) with fp16, batch_size=8.

In [ ]:
# --- Standard library imports ---
import logging
import sys
import time
from pathlib import Path

# --- Third-party imports ---
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# --- Project imports ---
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.languages import Language
from src.data.flores_loader import load_flores_parallel_corpus
from src.analysis.cross_lingual_embedding_alignment.hooks import load_model, ActivationStore, register_model_hooks
from src.analysis.cross_lingual_embedding_alignment.cross_lingual_alignment import CrossLingualAlignmentAnalyzer

# --- Logging ---
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# --- Configuration ---
# Change PRECISION to "4bit" if your GPU has less than 8GB VRAM.
PRECISION = "fp16"
BATCH_SIZE = 8
MAX_LENGTH = 128
MAX_SENTENCES = None  # Set to e.g., 100 for quick testing.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Output directories ---
RESULTS_DIR = PROJECT_ROOT / "results" / "cross_lingual"
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Precision: {PRECISION}")
print(f"Batch size: {BATCH_SIZE}")

## 1. Load Model

We load `CohereLabs/tiny-aya-global` — a 3.35B parameter multilingual
model with 4 transformer layers (3 sliding-window attention + 1
global attention). Hidden dimension is 3072.

In [ ]:
model, tokenizer = load_model(
    model_name="CohereLabs/tiny-aya-global",
    precision=PRECISION,
)

print(f"\nModel type: {type(model).__name__}")
print(f"Config: {model.config.num_hidden_layers} layers, "
      f"hidden_size={model.config.hidden_size}")

## 2. Load Parallel Corpus

In [ ]:
corpus = load_flores_parallel_corpus(max_sentences=MAX_SENTENCES)
n_sentences = len(next(iter(corpus.values())))
print(f"Loaded {len(corpus)} languages, {n_sentences} sentences each.")

## 3. Extract Activations

We use the `CrossLingualAlignmentAnalyzer` to orchestrate activation
extraction. For each language, all sentences are processed through
the model in batches, and mean-pooled hidden states are collected
at every layer.

**Memory note**: Activations are moved to CPU immediately after
extraction to keep GPU memory free for the next batch.

In [ ]:
analyzer = CrossLingualAlignmentAnalyzer(
    model=model,
    tokenizer=tokenizer,
    parallel_corpus=corpus,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE,
)

# Time the extraction.
start_time = time.time()
analyzer.extract_all_activations()
elapsed = time.time() - start_time

print(f"\nExtraction completed in {elapsed:.1f} seconds.")

## 4. Verify Activation Shapes

Confirm that we have the expected shapes: `(n_sentences, hidden_dim)`
for each language and layer.

In [ ]:
print("=== Activation Shapes ===")
for lang_name in sorted(analyzer.activations.keys())[:3]:
    print(f"\n{lang_name}:")
    for layer_name, tensor in sorted(analyzer.activations[lang_name].items()):
        print(f"  {layer_name}: {tuple(tensor.shape)}")

# Print summary.
sample_lang = next(iter(analyzer.activations))
sample_layer = next(iter(analyzer.activations[sample_lang]))
shape = analyzer.activations[sample_lang][sample_layer].shape
print(f"\nAll activations: {len(analyzer.activations)} languages x "
      f"{len(analyzer.activations[sample_lang])} layers x "
      f"{shape}")

## 5. Save Activations to Disk

Save all activations as `.pt` files with standardized naming:
`layer_{idx}_{language_name}.pt`

In [ ]:
analyzer.save_activations(str(RESULTS_DIR))
print(f"Activations saved to {RESULTS_DIR / 'activations'}")

## 6. Dimensionality Reduction Visualization

Visualize the embedding spaces at each layer using PCA (for variance
analysis) and t-SNE (for cluster visualization). Points are colored
by language.

**What to look for**:
- **Early layers**: Languages should form distinct clusters (language-
  specific representations).
- **Later layers**: Clusters should merge (language-agnostic
  representations emerging).

In [ ]:
# Use a subset of sentences for visualization (t-SNE is slow on 13K+ points).
N_VIS = min(200, n_sentences)

# Collect embeddings for visualization.
all_embeddings = {}  # {layer_idx: (n_langs * N_VIS, d)}
all_labels = []      # Language labels for coloring.

for layer_idx in analyzer.layer_indices:
    layer_name = f"layer_{layer_idx}"
    layer_embs = []

    for lang in analyzer.languages:
        emb = analyzer.activations[lang.lang_name][layer_name][:N_VIS]
        layer_embs.append(emb.numpy())

    all_embeddings[layer_idx] = np.concatenate(layer_embs, axis=0)

# Build label array (same for all layers).
all_labels = []
for lang in analyzer.languages:
    all_labels.extend([lang.iso_code] * N_VIS)
all_labels = np.array(all_labels)

### 6a. PCA Visualization

In [ ]:
n_layers = len(analyzer.layer_indices)
fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 5))

if n_layers == 1:
    axes = [axes]

unique_langs = sorted(set(all_labels))
color_map = plt.cm.tab20(np.linspace(0, 1, len(unique_langs)))
lang_to_color = {lang: color_map[i] for i, lang in enumerate(unique_langs)}

for ax, layer_idx in zip(axes, analyzer.layer_indices):
    embeddings = all_embeddings[layer_idx]

    # Fit PCA to 2 components.
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(embeddings)

    # Plot each language as a separate scatter for legend support.
    for lang_iso in unique_langs:
        mask = all_labels == lang_iso
        ax.scatter(
            reduced[mask, 0],
            reduced[mask, 1],
            c=[lang_to_color[lang_iso]],
            label=lang_iso,
            s=8,
            alpha=0.5,
        )

    ax.set_title(
        f"Layer {layer_idx}\n"
        f"(PC1: {pca.explained_variance_ratio_[0]:.1%}, "
        f"PC2: {pca.explained_variance_ratio_[1]:.1%})",
        fontsize=11,
        fontweight="bold",
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

# Add legend to the last subplot.
axes[-1].legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=7,
    markerscale=2,
)

fig.suptitle(
    "PCA of Sentence Embeddings by Language",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pca_embeddings_by_layer.png", bbox_inches="tight")
plt.show()

### 6b. t-SNE Visualization

t-SNE is better at revealing local cluster structure. Expect to see
language-specific clusters in early layers that dissolve in later layers.

In [ ]:
fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 5))

if n_layers == 1:
    axes = [axes]

for ax, layer_idx in zip(axes, analyzer.layer_indices):
    embeddings = all_embeddings[layer_idx]

    # Fit t-SNE.
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    reduced = tsne.fit_transform(embeddings)

    for lang_iso in unique_langs:
        mask = all_labels == lang_iso
        ax.scatter(
            reduced[mask, 0],
            reduced[mask, 1],
            c=[lang_to_color[lang_iso]],
            label=lang_iso,
            s=8,
            alpha=0.5,
        )

    ax.set_title(f"Layer {layer_idx}", fontsize=11, fontweight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

axes[-1].legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=7,
    markerscale=2,
)

fig.suptitle(
    "t-SNE of Sentence Embeddings by Language",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "tsne_embeddings_by_layer.png", bbox_inches="tight")
plt.show()

### 6c. PCA Variance Explained per Layer

Track how many principal components are needed to explain 90% of
variance at each layer. A decrease suggests representations are
becoming lower-dimensional (more compressed).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for layer_idx in analyzer.layer_indices:
    embeddings = all_embeddings[layer_idx]
    pca = PCA(n_components=min(50, embeddings.shape[1]))
    pca.fit(embeddings)

    cumulative_var = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(
        range(1, len(cumulative_var) + 1),
        cumulative_var,
        "-o",
        markersize=3,
        label=f"Layer {layer_idx}",
    )

ax.axhline(y=0.9, color="red", linestyle="--", alpha=0.7, label="90% threshold")
ax.set_xlabel("Number of Principal Components", fontsize=12)
ax.set_ylabel("Cumulative Variance Explained", fontsize=12)
ax.set_title("PCA Variance Explained per Layer", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "pca_variance_explained.png", bbox_inches="tight")
plt.show()

## 7. Summary

**Key observations:**

- Activations successfully extracted for all 13 languages x 4 layers.
- Activations saved to disk for reuse in subsequent notebooks.
- PCA and t-SNE visualizations reveal the language clustering
  structure at each layer.
- Next: Compute cross-lingual CKA in Notebook 03.

In [ ]:
print("\n=== Activation Extraction Complete ===")
print(f"  Extraction time: {elapsed:.1f}s")
print(f"  Activations saved: {RESULTS_DIR / 'activations'}")
print(f"  Figures saved: {FIGURES_DIR}")
print("\nNext step: Run Notebook 03 for cross-lingual CKA analysis.")